# 11. Overfitting

**Statistical Foundations for Data Science — Notebook 11 of 12**

**Overfitting** is when a model learns the accidents of your training data instead of the
pattern underneath. It shows up as a large gap between training and validation performance,
and it is the default failure mode of every flexible model.

Its mirror image is **underfitting**: the model is too rigid to represent the pattern at
all, so it does badly on *both* training and validation data.

Almost all of applied machine learning is navigating between these two.

### What you will learn

1. Overfitting vs underfitting: how to tell them apart from two numbers
2. **Model capacity** and where it comes from
3. **Learning curves** — diagnosing whether you need a better model or more data
4. **Validation curves** — finding the right complexity
5. The cures: more data, fewer features, **regularisation** ($L_1$, $L_2$), early stopping,
   pruning, ensembling, cross-validation
6. Overfitting in classification: decision boundaries
7. Overfitting you cannot see: leakage, repeated testing, and the small-data trap
8. A practical diagnosis-to-treatment playbook

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (train_test_split, cross_val_score, learning_curve,
                                     validation_curve, KFold, StratifiedKFold)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_moons, make_regression

rng = np.random.default_rng(seed=11)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)

---
## 11.1 The two failure modes

| | Training error | Validation error | Diagnosis | What to do |
|---|---|---|---|---|
| **Underfitting** | high | high (≈ training) | model too simple / features too weak | more capacity, better features, less regularisation |
| **Good fit** | low | low (≈ training) | — | ship it |
| **Overfitting** | very low | much higher | model memorised the noise | more data, less capacity, more regularisation |
| **Something is wrong** | high | *lower* than training | leakage, a bug, or a tiny validation set | investigate before celebrating |

The diagnostic is the **gap**, not either number alone. A 2% training error with 3%
validation error is a good model. A 0% training error with 25% validation error is a
memoriser.

In [ ]:
# The same three fits from Notebook 10, now labelled by diagnosis
def true_f(t):
    return np.sin(1.4 * t) + 0.25 * t

n = 40
x_all = rng.uniform(0, 7, n)
y_all = true_f(x_all) + rng.normal(0, 0.35, n)
X = x_all.reshape(-1, 1)
X_tr, X_va, y_tr, y_va = train_test_split(X, y_all, test_size=0.35, random_state=1)

rows = []
for d in (1, 2, 4, 9, 15):
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(X_tr, y_tr)
    tr = np.sqrt(mean_squared_error(y_tr, m.predict(X_tr)))
    va = np.sqrt(mean_squared_error(y_va, m.predict(X_va)))
    if va > 2.2 * tr:
        verdict = "OVERFIT"
    elif tr > 0.55:
        verdict = "UNDERFIT"
    else:
        verdict = "good fit"
    rows.append({"degree": d, "train_RMSE": round(tr, 4), "val_RMSE": round(va, 4),
                 "gap": round(va - tr, 4), "ratio": round(va / tr, 2), "diagnosis": verdict})
print(pd.DataFrame(rows).to_string(index=False))
print("\nNotice the degree-15 row: essentially zero training error, terrible validation.")
print("That is the fingerprint of memorisation.")

---
## 11.2 Where capacity comes from

**Capacity** (or flexibility) is how many distinct functions a model can represent. More
capacity means it can fit more complicated truths — and more noise.

| Model | Capacity increases with |
|---|---|
| Polynomial regression | degree |
| Linear models | number of features; smaller regularisation |
| Decision tree | depth, leaves; smaller `min_samples_leaf` |
| k-NN | **smaller** k ($k=1$ is maximum capacity) |
| SVM (RBF) | larger `C`, larger `gamma` |
| Neural network | layers, units, training epochs |
| Random forest / boosting | more/deeper trees (though forests resist overfitting) |

The other half of the equation is **how much data you have**. Capacity is only dangerous
relative to $n$. A degree-15 polynomial on 40 points is reckless; on 40,000 points it is
almost harmless.

In [ ]:
# The same model complexity, three dataset sizes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
grid = np.linspace(0, 7, 400).reshape(-1, 1)
DEG = 12
for ax, m_size in zip(axes, [15, 60, 600]):
    xs = rng.uniform(0, 7, m_size)
    ys = true_f(xs) + rng.normal(0, 0.35, m_size)
    Xa, Xb, ya, yb = train_test_split(xs.reshape(-1, 1), ys, test_size=0.3, random_state=0)
    mdl = make_pipeline(PolynomialFeatures(DEG), LinearRegression()).fit(Xa, ya)
    ax.plot(grid, true_f(grid.ravel()), "k--", lw=1.5, label="truth")
    ax.plot(grid, mdl.predict(grid), color="crimson", lw=2, label=f"degree {DEG} fit")
    ax.scatter(Xa, ya, s=18, color="steelblue", alpha=0.7)
    ax.set_ylim(-3, 4); ax.legend(fontsize=7)
    ax.set_title(f"n = {m_size}\ntrain RMSE {np.sqrt(mean_squared_error(ya, mdl.predict(Xa))):.3f}, "
                 f"val {np.sqrt(mean_squared_error(yb, mdl.predict(Xb))):.3f}", fontsize=9)
plt.tight_layout(); plt.show()

print("Identical capacity. The only thing that changed is n -- and with enough data the")
print("degree-12 polynomial stops hallucinating and tracks the truth.")
print("\nThis is why 'get more data' is the most reliable cure for overfitting.")

---
## 11.3 Validation curves: choosing complexity

A **validation curve** plots training and cross-validated scores against one hyperparameter.
The classic shape:

- Left side: both scores poor → **underfitting**
- Middle: validation score peaks → **the sweet spot**
- Right side: training score keeps improving while validation degrades → **overfitting**

`sklearn.model_selection.validation_curve` does the sweep for you.

In [ ]:
degrees = np.arange(1, 16)
train_sc, val_sc = validation_curve(
    make_pipeline(PolynomialFeatures(), Ridge(alpha=1e-8)),
    X, y_all,
    param_name="polynomialfeatures__degree", param_range=degrees,
    cv=KFold(5, shuffle=True, random_state=0),
    scoring="neg_root_mean_squared_error",
)
train_rmse, val_rmse = -train_sc.mean(axis=1), -val_sc.mean(axis=1)
val_sd = val_sc.std(axis=1)

plt.plot(degrees, train_rmse, "o-", color="steelblue", label="training RMSE")
plt.plot(degrees, val_rmse, "o-", color="crimson", label="cross-validated RMSE")
plt.fill_between(degrees, val_rmse - val_sd, val_rmse + val_sd, color="crimson", alpha=0.15)
best_d = degrees[int(np.argmin(val_rmse))]
plt.axvline(best_d, color="black", ls="--", label=f"best degree = {best_d}")
plt.yscale("log"); plt.xlabel("polynomial degree"); plt.ylabel("RMSE (log scale)")
plt.title("Validation curve: underfit on the left, overfit on the right")
plt.legend(fontsize=8); plt.show()

print(f"{'degree':>7}{'train':>10}{'CV':>10}{'CV sd':>9}")
for d, a, b_, c_ in zip(degrees, train_rmse, val_rmse, val_sd):
    print(f"{d:>7}{a:>10.4f}{b_:>10.4f}{c_:>9.4f}{'  <- best' if d == best_d else ''}")

# The one-standard-error rule: prefer the simplest model within 1 SE of the best
thresh = val_rmse.min() + val_sd[int(np.argmin(val_rmse))]
simplest = degrees[np.argmax(val_rmse <= thresh)]
print(f"\nOne-standard-error rule: the simplest degree within 1 SE of the best is {simplest}.")
print("Preferring it trades a hair of accuracy for a more stable, more interpretable model.")

In [ ]:
# The same diagnostic for a decision tree, where the knob is depth
depths = np.arange(1, 21)
tr_s, va_s = validation_curve(DecisionTreeRegressor(random_state=0), X, y_all,
                              param_name="max_depth", param_range=depths,
                              cv=KFold(5, shuffle=True, random_state=0),
                              scoring="neg_root_mean_squared_error")
plt.plot(depths, -tr_s.mean(axis=1), "o-", color="steelblue", label="training")
plt.plot(depths, -va_s.mean(axis=1), "o-", color="crimson", label="cross-validated")
plt.xlabel("max_depth"); plt.ylabel("RMSE")
plt.title("Deeper trees memorise: training error goes to zero, CV error plateaus")
plt.legend(); plt.show()

print(f"Training RMSE at depth 20 : {-tr_s.mean(axis=1)[-1]:.6f}")
print(f"CV RMSE at depth 20       : {-va_s.mean(axis=1)[-1]:.4f}")
print(f"Best depth by CV          : {depths[int(np.argmax(va_s.mean(axis=1)))]}")

---
## 11.4 Learning curves: do you need more data or a better model?

A **learning curve** plots performance against **training set size**. It answers the single
most valuable question in a stalled project:

| Shape | Meaning | Action |
|---|---|---|
| Curves converge at a **poor** score | **High bias** — the model is too simple | more features, more capacity, less regularisation. **More data will not help.** |
| A large **persistent gap**, validation still improving | **High variance** — overfitting | **more data will help**; or regularise / simplify |
| Curves converge at a **good** score | You are done | stop |

This is the diagnostic that stops teams from spending six months labelling data that cannot
help them.

In [ ]:
X_big, y_big = make_regression(n_samples=1500, n_features=20, n_informative=8,
                               noise=25.0, random_state=1)

def show_learning_curve(estimator, X_, y_, title, ax):
    sizes, tr, va = learning_curve(estimator, X_, y_,
                                   train_sizes=np.linspace(0.05, 1.0, 12),
                                   cv=KFold(5, shuffle=True, random_state=0),
                                   scoring="neg_root_mean_squared_error")
    tr, va = -tr, -va
    ax.plot(sizes, tr.mean(axis=1), "o-", color="steelblue", label="training")
    ax.fill_between(sizes, tr.mean(1)-tr.std(1), tr.mean(1)+tr.std(1),
                    color="steelblue", alpha=0.15)
    ax.plot(sizes, va.mean(axis=1), "o-", color="crimson", label="cross-validated")
    ax.fill_between(sizes, va.mean(1)-va.std(1), va.mean(1)+va.std(1),
                    color="crimson", alpha=0.15)
    ax.set_xlabel("training set size"); ax.set_ylabel("RMSE")
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8)
    return sizes, tr.mean(axis=1), va.mean(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
a = show_learning_curve(make_pipeline(StandardScaler(), Ridge(alpha=1e6)), X_big, y_big,
                        "HIGH BIAS\n(over-regularised linear model)", axes[0])
b = show_learning_curve(make_pipeline(StandardScaler(), Ridge(alpha=1.0)), X_big, y_big,
                        "GOOD FIT\n(sensible linear model)", axes[1])
c = show_learning_curve(DecisionTreeRegressor(random_state=0), X_big, y_big,
                        "HIGH VARIANCE\n(unpruned decision tree)", axes[2])
plt.tight_layout(); plt.show()

In [ ]:
for name, (sizes, tr, va) in [("high bias", a), ("good fit", b), ("high variance", c)]:
    print(f"{name:<15} final train RMSE {tr[-1]:8.3f}   final CV RMSE {va[-1]:8.3f}   "
          f"gap {va[-1]-tr[-1]:7.3f}")
print()
print("Reading the three panels:")
print("  * high bias    : both curves flatten high and TOGETHER -> more rows change nothing")
print("  * good fit     : curves meet at a low value -> the model matches the problem")
print("  * high variance: training near zero, CV far above, still falling -> more rows help")
print()
print("Extrapolating the third curve tells you roughly how much more data is worth buying.")

---
## 11.5 Regularisation: capacity you can dial down

Regularisation adds a penalty on coefficient size to the loss, so the model must *earn* every
unit of complexity.

**Ridge ($L_2$)** — penalise squared coefficients:

$$\min_{\beta}\ \sum(y_i - \hat{y}_i)^2 + \alpha\sum_{j}\beta_j^2$$

Shrinks all coefficients smoothly toward zero (never exactly zero). Handles correlated
predictors gracefully — it splits the credit between them.

**Lasso ($L_1$)** — penalise absolute coefficients:

$$\min_{\beta}\ \sum(y_i - \hat{y}_i)^2 + \alpha\sum_{j}|\beta_j|$$

Drives some coefficients **exactly to zero**, so it performs feature selection.

**Elastic Net** — a mix of both, for when you want sparsity *and* stability.

$\alpha = 0$ is plain OLS; $\alpha \to \infty$ predicts the mean. Choose $\alpha$ by
cross-validation, never by eye. **Always scale your features first** — the penalty treats
all coefficients on the same footing, so their units must match.

In [ ]:
# 30 features, only 5 of which matter, and only 60 rows: OLS has no chance
m_r, p_r, k_true = 60, 30, 5
Xr = rng.normal(size=(m_r, p_r))
true_beta = np.zeros(p_r)
true_beta[:k_true] = [4.0, -3.0, 2.5, 2.0, -1.5]
yr = Xr @ true_beta + rng.normal(0, 1.5, m_r)

Xa, Xb, ya, yb = train_test_split(Xr, yr, test_size=0.35, random_state=0)

results = []
for name, est in [("OLS (no penalty)", LinearRegression()),
                  ("Ridge alpha=1",    Ridge(alpha=1.0)),
                  ("Ridge alpha=10",   Ridge(alpha=10.0)),
                  ("Lasso alpha=0.1",  Lasso(alpha=0.1, max_iter=20000)),
                  ("Lasso alpha=0.5",  Lasso(alpha=0.5, max_iter=20000))]:
    pipe = make_pipeline(StandardScaler(), est).fit(Xa, ya)
    coefs = pipe[-1].coef_
    results.append({
        "model": name,
        "train_RMSE": round(np.sqrt(mean_squared_error(ya, pipe.predict(Xa))), 3),
        "test_RMSE": round(np.sqrt(mean_squared_error(yb, pipe.predict(Xb))), 3),
        "nonzero_coefs": int((np.abs(coefs) > 1e-8).sum()),
        "max_|coef|": round(np.abs(coefs).max(), 2),
    })
print(pd.DataFrame(results).to_string(index=False))
print(f"\n(only {k_true} of the {p_r} features actually matter)")
print("OLS fits the training set best and generalises worst -- the textbook trade.")
print("Lasso recovers a sparse model close to the truth.")

In [ ]:
# Coefficient paths: watch shrinkage happen
alphas = np.logspace(-3, 2.5, 60)
ridge_path = np.array([make_pipeline(StandardScaler(), Ridge(alpha=al)).fit(Xa, ya)[-1].coef_
                       for al in alphas])
lasso_path = np.array([make_pipeline(StandardScaler(), Lasso(alpha=al, max_iter=20000))
                       .fit(Xa, ya)[-1].coef_ for al in alphas])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for j in range(p_r):
    style = dict(lw=2.0, alpha=0.95) if j < k_true else dict(lw=0.8, alpha=0.35)
    ax[0].plot(alphas, ridge_path[:, j], color="crimson" if j < k_true else "grey", **style)
    ax[1].plot(alphas, lasso_path[:, j], color="crimson" if j < k_true else "grey", **style)
for a_, t_ in zip(ax, ["Ridge (L2): everything shrinks, nothing vanishes",
                       "Lasso (L1): irrelevant coefficients hit exactly zero"]):
    a_.set_xscale("log"); a_.axhline(0, color="black", lw=0.8)
    a_.set_xlabel("alpha (penalty strength)"); a_.set_ylabel("coefficient")
    a_.set_title(t_, fontsize=10)
plt.tight_layout(); plt.show()
print("Red = the 5 genuinely useful features, grey = the 25 noise features.")

In [ ]:
# Choosing alpha properly: cross-validation
from sklearn.linear_model import RidgeCV, LassoCV

ridge_cv = make_pipeline(StandardScaler(),
                         RidgeCV(alphas=np.logspace(-3, 3, 100),
                                 cv=KFold(5, shuffle=True, random_state=0))).fit(Xa, ya)
lasso_cv = make_pipeline(StandardScaler(),
                         LassoCV(alphas=np.logspace(-3, 1, 100), max_iter=20000,
                                 cv=KFold(5, shuffle=True, random_state=0))).fit(Xa, ya)

print(f"Ridge: alpha chosen by CV = {ridge_cv[-1].alpha_:.4f}, "
      f"test RMSE = {np.sqrt(mean_squared_error(yb, ridge_cv.predict(Xb))):.3f}")
print(f"Lasso: alpha chosen by CV = {lasso_cv[-1].alpha_:.4f}, "
      f"test RMSE = {np.sqrt(mean_squared_error(yb, lasso_cv.predict(Xb))):.3f}")

sel = np.where(np.abs(lasso_cv[-1].coef_) > 1e-8)[0]
print(f"\nFeatures Lasso kept : {sel.tolist()}")
print(f"Features that matter: {list(range(k_true))}")
print(f"\nEstimated coefficients vs truth (first 8 features):")
for j in range(8):
    print(f"  x{j:<2} true {true_beta[j]:+6.2f}   ridge {ridge_cv[-1].coef_[j]:+6.2f}"
          f"   lasso {lasso_cv[-1].coef_[j]:+6.2f}")

---
## 11.6 Overfitting in classification: look at the boundary

For classification the symptom is a **decision boundary** that contorts itself around
individual training points instead of following the shape of the classes.

In [ ]:
Xm, ym = make_moons(n_samples=250, noise=0.32, random_state=3)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.35, random_state=0,
                                              stratify=ym)

def plot_boundary(model, ax, title):
    h = 0.02
    x_min, x_max = Xm[:, 0].min() - 0.5, Xm[:, 0].max() + 0.5
    y_min, y_max = Xm[:, 1].min() - 0.5, Xm[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(Xm_tr[:, 0], Xm_tr[:, 1], c=ym_tr, cmap="coolwarm", s=22,
               edgecolor="k", linewidth=0.3)
    tr = model.score(Xm_tr, ym_tr); te = model.score(Xm_te, ym_te)
    ax.set_title(f"{title}\ntrain {tr:.3f} / test {te:.3f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    return tr, te

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
models = [
    (make_pipeline(StandardScaler(), LogisticRegression()), "Logistic (underfit)"),
    (KNeighborsClassifier(n_neighbors=25), "k-NN, k=25"),
    (KNeighborsClassifier(n_neighbors=1), "k-NN, k=1 (overfit)"),
    (DecisionTreeClassifier(random_state=0), "Full tree (overfit)"),
]
scores = []
for ax, (mdl, name) in zip(axes, models):
    mdl.fit(Xm_tr, ym_tr)
    scores.append((name,) + plot_boundary(mdl, ax, name))
plt.tight_layout(); plt.show()

print(f"{'model':<24}{'train':>8}{'test':>8}{'gap':>8}")
for name, tr, te in scores:
    print(f"{name:<24}{tr:>8.3f}{te:>8.3f}{tr-te:>8.3f}")
print("\nk=1 and the full tree both reach perfect training accuracy by drawing islands")
print("around individual points. Those islands are noise, and they cost test accuracy.")

In [ ]:
# Pruning a tree: the effect of min_samples_leaf
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, leaf in zip(axes, [1, 5, 15, 40]):
    t = DecisionTreeClassifier(min_samples_leaf=leaf, random_state=0).fit(Xm_tr, ym_tr)
    plot_boundary(t, ax, f"min_samples_leaf = {leaf}")
plt.tight_layout(); plt.show()

print("Requiring more samples per leaf forces the tree to describe groups, not individuals.")
print("This is regularisation for trees; max_depth and ccp_alpha do the same job.")

# Cost-complexity pruning, chosen by CV
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xm_tr, ym_tr)
alphas_ccp = path.ccp_alphas[:-1]
cv_means = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a_, random_state=0),
                            Xm_tr, ym_tr, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
            for a_ in alphas_ccp]
best_ccp = alphas_ccp[int(np.argmax(cv_means))]
pruned = DecisionTreeClassifier(ccp_alpha=best_ccp, random_state=0).fit(Xm_tr, ym_tr)
print(f"\nBest ccp_alpha by CV = {best_ccp:.5f}")
print(f"Pruned tree: train {pruned.score(Xm_tr, ym_tr):.3f}, test {pruned.score(Xm_te, ym_te):.3f}, "
      f"leaves {pruned.get_n_leaves()}")
full = DecisionTreeClassifier(random_state=0).fit(Xm_tr, ym_tr)
print(f"Full tree  : train {full.score(Xm_tr, ym_tr):.3f}, test {full.score(Xm_te, ym_te):.3f}, "
      f"leaves {full.get_n_leaves()}")

---
## 11.7 Ensembling and early stopping

**Averaging many overfitted models** cancels much of their individual noise. A single deep
tree overfits badly; a forest of them, each grown on a bootstrap sample with random feature
subsets, does not. This is why random forests are so forgiving — Notebook 6 of the ML module
covers the mechanism.

**Early stopping** applies to any model trained iteratively (gradient descent, boosting,
neural networks): monitor validation error each epoch and stop when it starts rising. The
number of iterations *is* a capacity knob.

In [ ]:
print("A single deep tree vs a forest of the same deep trees:")
for name, est in [("1 unpruned tree", DecisionTreeRegressor(random_state=0)),
                  ("10 trees",  RandomForestRegressor(n_estimators=10, random_state=0)),
                  ("100 trees", RandomForestRegressor(n_estimators=100, random_state=0)),
                  ("300 trees", RandomForestRegressor(n_estimators=300, random_state=0))]:
    est.fit(X_big[:800], y_big[:800])
    tr = np.sqrt(mean_squared_error(y_big[:800], est.predict(X_big[:800])))
    cv = -cross_val_score(est, X_big, y_big, cv=KFold(5, shuffle=True, random_state=0),
                          scoring="neg_root_mean_squared_error").mean()
    print(f"  {name:<16} train RMSE {tr:7.3f}   CV RMSE {cv:7.3f}   gap {cv-tr:7.3f}")
print("\nEvery member still memorises its own bootstrap sample (training RMSE stays low),")
print("but their errors are partly independent, so averaging cancels them.")
print("More trees never makes a forest worse -- unusual, and very convenient.")

In [ ]:
# Early stopping, shown with gradient boosting's staged predictions
from sklearn.ensemble import GradientBoostingRegressor

Xa2, Xb2, ya2, yb2 = train_test_split(X_big, y_big, test_size=0.3, random_state=0)
gb = GradientBoostingRegressor(n_estimators=600, learning_rate=0.05, max_depth=4,
                               random_state=0).fit(Xa2, ya2)

tr_curve = [np.sqrt(mean_squared_error(ya2, p)) for p in gb.staged_predict(Xa2)]
te_curve = [np.sqrt(mean_squared_error(yb2, p)) for p in gb.staged_predict(Xb2)]
best_iter = int(np.argmin(te_curve)) + 1

plt.plot(tr_curve, color="steelblue", label="training RMSE")
plt.plot(te_curve, color="crimson", label="validation RMSE")
plt.axvline(best_iter, color="black", ls="--", label=f"stop here: {best_iter} trees")
plt.xlabel("number of boosting iterations"); plt.ylabel("RMSE")
plt.title("Early stopping: more iterations eventually hurt")
plt.legend(fontsize=8); plt.show()

print(f"Best validation RMSE {min(te_curve):.3f} at iteration {best_iter}")
print(f"Validation RMSE at iteration 600: {te_curve[-1]:.3f}")
print(f"Training RMSE at iteration 600  : {tr_curve[-1]:.3f}  (still falling)")
print("\nsklearn can do this for you: pass n_iter_no_change and validation_fraction.")

---
## 11.8 Overfitting you cannot see on a validation curve

The dangerous cases are the ones where your validation score is *itself* contaminated.

1. **Leakage** (Notebook 10) — validation looks great, production fails
2. **Repeated evaluation** — every time you look at the validation set and change something,
   you fit to it a little. After 50 experiments your validation score is a training score.
3. **Feature selection on all the data** — the choice of features already used the
   validation rows
4. **Tiny validation sets** — the score is so noisy that you are selecting on noise
5. **Distribution shift** — the future is not drawn from the same distribution as your data;
   no amount of internal validation detects this

In [ ]:
# Overfitting the validation set by repeated tinkering, on data with no signal
m_o, p_o = 250, 60
Xo = rng.normal(size=(m_o, p_o))
yo = rng.integers(0, 2, m_o)                       # pure noise target

Xa3, Xrest, ya3, yrest = train_test_split(Xo, yo, test_size=0.5, random_state=0, stratify=yo)
Xval, Xtest, yval, ytest = train_test_split(Xrest, yrest, test_size=0.5, random_state=0,
                                            stratify=yrest)

best_val, best_cfg, history = 0.0, None, []
for trial in range(200):
    cols = rng.choice(p_o, size=int(rng.integers(2, 12)), replace=False)
    C = float(10 ** rng.uniform(-3, 2))
    mdl = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=500))
    mdl.fit(Xa3[:, cols], ya3)
    v = mdl.score(Xval[:, cols], yval)
    if v > best_val:
        best_val, best_cfg = v, (mdl, cols)
    history.append(best_val)

mdl, cols = best_cfg
print(f"After 200 experiments:")
print(f"  best VALIDATION accuracy = {best_val:.4f}")
print(f"  its TEST accuracy        = {mdl.score(Xtest[:, cols], ytest):.4f}")
print(f"  truth                    = 0.5000 (the target is random)")

plt.plot(history, color="steelblue")
plt.axhline(0.5, color="black", ls="--", label="true accuracy (chance)")
plt.xlabel("number of experiments tried"); plt.ylabel("best validation accuracy so far")
plt.title("Validation accuracy creeps up on pure noise as you keep tinkering")
plt.legend(); plt.show()
print("\nDefences: a locked test set, nested CV, pre-registering the experiment plan,")
print("and counting how many configurations you actually tried.")

In [ ]:
# Distribution shift: perfect validation, useless deployment
m_s = 400
x_train_dom = rng.uniform(0, 5, m_s)                  # training covers x in [0, 5]
y_train_dom = true_f(x_train_dom) + rng.normal(0, 0.2, m_s)
x_future = rng.uniform(5, 9, 200)                     # production sees x in [5, 9]
y_future = true_f(x_future) + rng.normal(0, 0.2, 200)

mdl = make_pipeline(PolynomialFeatures(8), Ridge(alpha=1e-4)).fit(
    x_train_dom.reshape(-1, 1), y_train_dom)

cv = -cross_val_score(make_pipeline(PolynomialFeatures(8), Ridge(alpha=1e-4)),
                      x_train_dom.reshape(-1, 1), y_train_dom,
                      cv=KFold(5, shuffle=True, random_state=0),
                      scoring="neg_root_mean_squared_error").mean()
future_rmse = np.sqrt(mean_squared_error(y_future, mdl.predict(x_future.reshape(-1, 1))))

print(f"Cross-validated RMSE on the training domain : {cv:.4f}   (excellent)")
print(f"RMSE on the shifted production domain       : {future_rmse:.4f}")

gx = np.linspace(0, 9, 400).reshape(-1, 1)
plt.plot(gx, true_f(gx.ravel()), "k--", lw=1.5, label="truth")
plt.plot(gx, mdl.predict(gx), color="crimson", lw=2, label="model")
plt.scatter(x_train_dom, y_train_dom, s=10, alpha=0.4, color="steelblue", label="training domain")
plt.scatter(x_future, y_future, s=10, alpha=0.5, color="darkorange", label="production domain")
plt.axvline(5, color="black", lw=1)
plt.ylim(-4, 5); plt.legend(fontsize=8)
plt.title("Cross-validation cannot warn you about data it has never seen")
plt.show()
print("\nThe cure is monitoring: track input distributions in production and re-train")
print("when they drift. See the MLOps module for how.")

---
## 11.9 The playbook

```
Measure training error and validation error.

Both HIGH, close together  ->  UNDERFITTING
    add features / interactions / polynomial terms
    increase capacity (deeper tree, smaller k, larger C)
    reduce regularisation
    check the features actually carry signal

Training LOW, validation MUCH higher  ->  OVERFITTING
    get more training data                    (most reliable)
    reduce capacity (prune, shallower, larger k)
    add regularisation (ridge / lasso / dropout / weight decay)
    remove weak or redundant features
    ensemble (bagging, random forest)
    early stopping for iterative models
    make sure you are not selecting on the validation set

Validation BETTER than training  ->  SUSPICIOUS
    check for leakage, a scoring bug, or a validation set that is too small
    (a small amount of this is normal when training uses dropout or noise)

Validation good, PRODUCTION bad  ->  LEAKAGE or DISTRIBUTION SHIFT
    audit feature availability at prediction time
    check for duplicate/grouped rows across splits
    compare training and production input distributions
```

---
## Exercises

**Exercise 1.** On the bundled digits dataset, fit k-NN for $k = 1, 3, 5, \dots, 49$.
Plot training and cross-validated accuracy, identify where overfitting occurs, and pick $k$
with the one-standard-error rule.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
from sklearn.datasets import load_digits

dg = load_digits()
Xd, yd = dg.data, dg.target
print(f"{Xd.shape[0]} images, {Xd.shape[1]} pixel features, {len(np.unique(yd))} classes\n")

ks = np.arange(1, 50, 2)
tr_d, va_d = validation_curve(make_pipeline(StandardScaler(), KNeighborsClassifier()),
                              Xd, yd, param_name="kneighborsclassifier__n_neighbors",
                              param_range=ks,
                              cv=StratifiedKFold(5, shuffle=True, random_state=0),
                              scoring="accuracy")
tr_m, va_m, va_s = tr_d.mean(1), va_d.mean(1), va_d.std(1)

plt.plot(ks, tr_m, "o-", color="steelblue", label="training accuracy")
plt.plot(ks, va_m, "o-", color="crimson", label="cross-validated accuracy")
plt.fill_between(ks, va_m - va_s, va_m + va_s, color="crimson", alpha=0.15)
plt.xlabel("k (fewer neighbours = more capacity)"); plt.ylabel("accuracy")
plt.title("k-NN validation curve on digits"); plt.legend(fontsize=8); plt.show()

best_i = int(np.argmax(va_m))
thresh = va_m[best_i] - va_s[best_i]
simplest_k = ks[np.max(np.where(va_m >= thresh))]      # larger k = simpler model
print(f"k=1  : train {tr_m[0]:.4f}, CV {va_m[0]:.4f}  <- perfect training score,")
print(f"       because every point is its own nearest neighbour. Classic overfitting.")
print(f"Best k by CV                  : {ks[best_i]} (CV {va_m[best_i]:.4f} +/- {va_s[best_i]:.4f})")
print(f"One-standard-error rule chooses: k = {simplest_k}")
print("\nOverfitting lives on the LEFT of this plot, because for k-NN small k = high capacity.")

**Exercise 2.** Take a dataset with 40 features where only 6 matter, and 80 rows. Compare
OLS, Ridge and Lasso with $\alpha$ chosen by cross-validation. Report test RMSE, how many
features each keeps, and how well Lasso recovers the true support.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
m2, p2, k2 = 80, 40, 6
X2 = rng.normal(size=(m2, p2))
beta2 = np.zeros(p2); beta2[:k2] = [5, -4, 3, -2.5, 2, -1.5]
y2 = X2 @ beta2 + rng.normal(0, 2.0, m2)

Xa4, Xb4, ya4, yb4 = train_test_split(X2, y2, test_size=0.3, random_state=0)
cv5 = KFold(5, shuffle=True, random_state=0)

fits = {
    "OLS":   make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 100), cv=cv5)),
    "Lasso": make_pipeline(StandardScaler(), LassoCV(alphas=np.logspace(-3, 1, 100),
                                                    max_iter=50000, cv=cv5)),
}
out = []
for name, pipe in fits.items():
    pipe.fit(Xa4, ya4)
    cf = pipe[-1].coef_
    kept = set(np.where(np.abs(cf) > 1e-8)[0])
    out.append({
        "model": name,
        "alpha": round(getattr(pipe[-1], "alpha_", np.nan), 4),
        "train_RMSE": round(np.sqrt(mean_squared_error(ya4, pipe.predict(Xa4))), 3),
        "test_RMSE": round(np.sqrt(mean_squared_error(yb4, pipe.predict(Xb4))), 3),
        "features_kept": len(kept),
        "true_features_found": len(kept & set(range(k2))),
        "false_positives": len(kept - set(range(k2))),
    })
print(pd.DataFrame(out).to_string(index=False))
print(f"\nTruth: {k2} informative features out of {p2}, n = {m2}")
print("OLS uses all 40 and overfits badly. Ridge shrinks but keeps all 40.")
print("Lasso recovers most of the true support and discards most of the noise --")
print("which is why it doubles as a feature-selection tool.")

**Exercise 3.** You are told a model has 99.2% cross-validated accuracy on fraud detection,
but it flags almost nothing useful in production. Diagnose it: produce a simulation showing
(a) how class imbalance inflates accuracy, and (b) how a leaked feature inflates it further,
then show what to measure instead.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score)
from sklearn.dummy import DummyClassifier

m3 = 20_000
fraud = (rng.random(m3) < 0.008).astype(int)          # 0.8% fraud
amount = rng.lognormal(4 + 0.8*fraud, 1.0, m3)
n_countries = rng.poisson(1 + 2*fraud, m3)
chargeback_filed = fraud * (rng.random(m3) < 0.9)     # LEAK: only known after the fact

Xf = pd.DataFrame({"amount": amount, "n_countries": n_countries})
Xf_leak = Xf.assign(chargeback_filed=chargeback_filed.astype(int))

def evaluate(Xdata, label):
    Xa5, Xb5, ya5, yb5 = train_test_split(Xdata, fraud, test_size=0.3, random_state=0,
                                          stratify=fraud)
    mdl = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xa5, ya5)
    pred = mdl.predict(Xb5)
    prob = mdl.predict_proba(Xb5)[:, 1]
    print(f"\n{label}")
    print(f"  accuracy          {accuracy_score(yb5, pred):.4f}")
    print(f"  precision         {precision_score(yb5, pred, zero_division=0):.4f}")
    print(f"  recall            {recall_score(yb5, pred, zero_division=0):.4f}")
    print(f"  F1                {f1_score(yb5, pred, zero_division=0):.4f}")
    print(f"  ROC-AUC           {roc_auc_score(yb5, prob):.4f}")
    print(f"  avg precision (PR){average_precision_score(yb5, prob):.4f}")
    print(f"  confusion matrix  {confusion_matrix(yb5, pred).ravel().tolist()} "
          f"[tn, fp, fn, tp]")

Xa5, Xb5, ya5, yb5 = train_test_split(Xf, fraud, test_size=0.3, random_state=0, stratify=fraud)
dummy = DummyClassifier(strategy="most_frequent").fit(Xa5, ya5)
print(f"(a) 'Always predict not-fraud' baseline accuracy = {dummy.score(Xb5, yb5):.4f}")
print("    With 0.8% positives, 99.2% accuracy is what you get for doing NOTHING.")
print("    Accuracy is the wrong metric for imbalanced problems.")

evaluate(Xf, "Legitimate features only:")
evaluate(Xf_leak, "(b) With the leaked chargeback feature:")

print("\nWhat to measure instead:")
print("  * precision and recall at the operating threshold the business will use")
print("  * average precision / PR-AUC, which ignores the huge true-negative mass")
print("  * cost-weighted metrics: what does a missed fraud cost vs a false alarm?")
print("  * always compare against a DummyClassifier baseline")

**Exercise 4 (challenge).** Build the full diagnosis-and-treatment loop on one dataset:
start from a badly overfitted model, use a learning curve to decide whether more data or
less capacity is the answer, apply the fix, and quantify the improvement. Justify each step
with a number.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
Xw, yw = make_regression(n_samples=700, n_features=25, n_informative=6,
                         noise=30.0, random_state=5)
Xa6, Xte6, ya6, yte6 = train_test_split(Xw, yw, test_size=0.25, random_state=0)
cv6 = KFold(5, shuffle=True, random_state=0)

def report(est, name):
    est.fit(Xa6, ya6)
    tr = np.sqrt(mean_squared_error(ya6, est.predict(Xa6)))
    cvv = -cross_val_score(est, Xa6, ya6, cv=cv6,
                           scoring="neg_root_mean_squared_error").mean()
    te = np.sqrt(mean_squared_error(yte6, est.predict(Xte6)))
    print(f"  {name:<38} train {tr:7.2f}   CV {cvv:7.2f}   test {te:7.2f}   gap {cvv-tr:7.2f}")
    return tr, cvv, te

print("STEP 1 - baseline, deliberately overfitted (degree-2 expansion, no penalty):")
base = make_pipeline(PolynomialFeatures(2), StandardScaler(), LinearRegression())
b_tr, b_cv, b_te = report(base, "PolyFeatures(2) + OLS")
print(f"  {base.fit(Xa6, ya6)[0].fit_transform(Xa6).shape[1]} engineered features for "
      f"{len(ya6)} training rows -> the gap is the diagnosis.")

In [ ]:
print("STEP 2 - learning curve: is the answer more data, or less capacity?")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sizes, tr_c, va_c = show_learning_curve(base, Xa6, ya6, "Overfitted baseline", ax[0])
_, tr_r, va_r = show_learning_curve(
    make_pipeline(PolynomialFeatures(2), StandardScaler(), Ridge(alpha=50)),
    Xa6, ya6, "After regularisation", ax[1])
plt.tight_layout(); plt.show()

print(f"  Baseline: gap at the largest training size = {va_c[-1] - tr_c[-1]:.2f}, "
      f"and the CV curve is still falling")
print("  -> a large, shrinking gap means HIGH VARIANCE: both more data AND")
print("     regularisation should help. Regularisation is free; data is not.")
print(f"  Regularised: gap = {va_r[-1] - tr_r[-1]:.2f}  (much smaller)")

In [ ]:
print("STEP 3 - apply the treatments and measure each one:")
report(base, "baseline: PolyFeatures(2) + OLS")
report(make_pipeline(StandardScaler(), LinearRegression()),
       "reduce capacity: drop the expansion")
report(make_pipeline(PolynomialFeatures(2), StandardScaler(),
                     RidgeCV(alphas=np.logspace(-2, 4, 100), cv=cv6)),
       "regularise: PolyFeatures(2) + RidgeCV")
report(make_pipeline(PolynomialFeatures(2), StandardScaler(),
                     LassoCV(alphas=np.logspace(-2, 3, 60), max_iter=50000, cv=cv6)),
       "sparsify: PolyFeatures(2) + LassoCV")
report(RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0),
       "ensemble: random forest")

print("\nSTEP 4 - conclusion")
print("  The winning model is the one with the lowest CV RMSE; the test column confirms")
print("  it without having been used to choose. Report the test number, the CV spread,")
print("  and the train-CV gap together -- that triple tells the whole story.")
print("\n  Note also what did NOT work: adding capacity. When the gap is the problem,")
print("  the answer is always constraint or data, never more flexibility.")

---
## Summary

| Concept | Key point |
|---|---|
| Overfitting | Low training error, high validation error — the model learned noise |
| Underfitting | Both errors high and close — the model is too rigid |
| The diagnostic | The **gap**, not either number alone |
| Capacity | Degree, depth, features, $1/k$, $C$, epochs — always relative to $n$ |
| Validation curve | Score vs one hyperparameter → find the sweet spot |
| Learning curve | Score vs training-set size → more data, or a better model? |
| Ridge ($L_2$) | Shrinks all coefficients; good with correlated features |
| Lasso ($L_1$) | Zeroes coefficients; doubles as feature selection |
| Scaling | Mandatory before any penalty |
| Pruning | `max_depth`, `min_samples_leaf`, `ccp_alpha` for trees |
| Ensembling | Averaging overfitted models cancels their independent errors |
| Early stopping | Number of iterations is a capacity knob |
| One-standard-error rule | Prefer the simplest model within 1 SE of the best |
| Invisible overfitting | Leakage, repeated evaluation, tiny validation sets, drift |

**Next up:** [Notebook 12 — Bias-Variance Tradeoff](12.%20Bias-Variance%20Tradeoff.ipynb),
which gives the mathematical account of *why* this trade-off exists at all.